# Maia Lite Chat 0.3.0 — Corpus Curation & Readiness v6

**Goal:** turn the v5 candidate inventory into a reproducible, rights-gated research corpus, without silently approving sources or restarting downloads.

This notebook:
- Reuses `corpus_v5` reports and existing `corpus_v4/inbox` files.
- Creates an **editable source-level rights review** with explicit approval evidence and attribution.
- Reports potential vs approved tokens, quality issues, coverage, source diversity, exact/near-duplicates and optional benchmark overlap.
- Creates **source-level train/validation/test splits** and exports **only approved** records.
- Supports separately imported authentic dialogue/educational files.
- Provides an **optional** conservative MediaWiki retry helper (not executed automatically).
- Provides an **optional** tokenizer comparison using approved training text only.

**Important:** Approval is a human decision. The code validates the review record; it does not make legal determinations. Counts are GPT-2 proxy estimates until a final tokenizer is trained.

In [ ]:
%pip -q install "transformers>=4.44,<6" "tokenizers>=0.19" pandas requests

In [ ]:
# 2. Mount Drive and configure paths
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, hashlib, json, random, re, unicodedata, time
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not in Colab: using local path if present.")

ROOT = Path("/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat")
V5 = ROOT / "corpus_v5"
V4_INBOX = ROOT / "corpus_v4/inbox"
V6 = ROOT / "corpus_v6"
REPORT = V6 / "reports"
EXPORT = V6 / "exports"
IMPORT = V6 / "imports"
for folder in (REPORT, EXPORT, IMPORT):
    folder.mkdir(parents=True, exist_ok=True)

V5_MANIFEST = V5 / "reports/maia_corpus_v030_manifest_v5.json"
if not V5_MANIFEST.exists():
    raise FileNotFoundError(f"Run v5 first: {V5_MANIFEST}")
manifest5 = json.loads(V5_MANIFEST.read_text(encoding="utf-8"))
print("v5 sources:", len(manifest5.get("sources", [])))
print("v5 potential tokens:", manifest5.get("potential_external_tokens_proxy"))
print("v5 approved tokens:", manifest5.get("approved_external_tokens_proxy"))

## A. Source registry and review workflow

Review file: `corpus_v6/reports/source_rights_review_v6.json`.

For each source, supply a verifiable license/rights basis, review date, reviewer, source URL, attribution and redistribution/training-use decision. **Do not approve based only on a filename or because a file is on Gutenberg/Wikimedia.** Review translations, editions, exceptions and obligations. The notebook keeps pending sources available for auditing but **never exports them as training text**.

To add new dialogue/educational texts, place UTF-8 `.txt` files under `corpus_v6/imports/<pt|en|es>/<dialogue|educational|language|creative|logic>/<collection>/`. One file should correspond to one independently reviewable work/source; include provenance in the review record.

In [ ]:
# 3. Discover existing + imported files; never silently replace the v5 inventory
ALLOWED_LANG = {"pt", "en", "es"}
ALLOWED_DOMAIN = {"language", "dialogue", "logic", "educational", "creative"}
records = {}
for item in manifest5.get("sources", []):
    key = item.get("source")
    p = Path(item.get("source_path", ""))
    if key and p.is_file() and p.suffix.lower() in {".txt", ".jsonl"}:
        parts = Path(key).parts
        if len(parts) >= 3 and parts[0] in ALLOWED_LANG and parts[1] in ALLOWED_DOMAIN:
            records[key] = {"path": str(p), "lang": parts[0], "domain": parts[1],
                            "v5_tokens_proxy": item.get("tokens_proxy_unique", 0)}
for p in sorted(IMPORT.rglob("*")):
    if not p.is_file() or p.suffix.lower() not in {".txt", ".jsonl"}:
        continue
    rel = p.relative_to(IMPORT)
    if len(rel.parts) < 4 or rel.parts[0] not in ALLOWED_LANG or rel.parts[1] not in ALLOWED_DOMAIN:
        print("SKIPPED invalid import path:", rel)
        continue
    key = "imports/" + rel.as_posix()
    records[key] = {"path": str(p), "lang": rel.parts[0], "domain": rel.parts[1],
                    "v5_tokens_proxy": None}
print("Discovered", len(records), "sources")
print(pd.DataFrame([{"source": k, **v} for k,v in records.items()])[["source","lang","domain"]].to_string(index=False))

In [ ]:
# 4. Create/preserve review ledger; NEVER overwrite existing human decisions
LEDGER = REPORT / "source_rights_review_v6.json"
if LEDGER.exists():
    reviews = json.loads(LEDGER.read_text(encoding="utf-8"))
else:
    reviews = {}
for key, info in records.items():
    reviews.setdefault(key, {
        "status": "pending_review",
        "review_complete": False,
        "approved_for_training": False,
        "reviewer": "",
        "reviewed_utc": "",
        "source_url": "",
        "rights_basis": "",
        "license_identifier": "",
        "attribution": "",
        "obligations": "",
        "translation_edition_checked": False,
        "training_use_checked": False,
        "redistribution_checked": False,
        "notes": ""
    })
LEDGER.write_text(json.dumps(reviews, indent=2, ensure_ascii=False), encoding="utf-8")
print("Review/edit in Drive:", LEDGER)
print("Pending:", sum(x.get("status") == "pending_review" for x in reviews.values()))

In [ ]:
# 5. Validate approval decisions (edit the JSON, then rerun this cell onward)
reviews = json.loads(LEDGER.read_text(encoding="utf-8"))
def review_valid(r):
    if r.get("status") != "approved" or r.get("approved_for_training") is not True:
        return False
    required = ("reviewer", "reviewed_utc", "source_url", "rights_basis",
                "license_identifier", "attribution", "obligations")
    if not r.get("review_complete") or not all(str(r.get(k, "")).strip() for k in required):
        return False
    return all(r.get(k) is True for k in
               ("translation_edition_checked", "training_use_checked", "redistribution_checked"))

approved = {key for key in records if review_valid(reviews.get(key, {}))}
invalid_approvals = [key for key in records if reviews.get(key, {}).get("approved_for_training")
                     and key not in approved]
if invalid_approvals:
    print("INVALID approvals (will be excluded):", invalid_approvals)
print("Eligible sources:", len(approved), "/", len(records))

## B. Cleaning, audits and candidate inventory

- Unicode NFC, whitespace normalization and Gutenberg boilerplate stripping.
- Source-level exact duplicate and conservative near-duplicate checks.
- Language/domain and quality flags are **diagnostics**, not silent exclusions.
- Wikipedia JSONL parsing uses a configurable text field. Inspect unexpected JSON structures rather than counting JSON markup as language.

In [ ]:
# 6. Parse, clean and segment
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("gpt2")
TEXT_KEYS = ("text", "content", "extract", "body")
def read_documents(path):
    if path.suffix == ".txt":
        yield path.read_text(encoding="utf-8", errors="replace")
    else:
        with path.open(encoding="utf-8", errors="replace") as handle:
            for line in handle:
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if isinstance(obj, str):
                    yield obj
                elif isinstance(obj, dict):
                    value = next((obj[k] for k in TEXT_KEYS if isinstance(obj.get(k), str)), None)
                    if value:
                        yield value

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    start = re.search(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG", text, re.I)
    end = re.search(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG", text, re.I)
    if start and end and start.end() < end.start():
        text = text[start.end():end.start()]
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{4,}", "\n\n", text)
    return text.strip()

def segments(text, max_chars=1800):
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = paragraph.strip()
        if len(paragraph) < 80:
            continue
        for start in range(0, len(paragraph), max_chars):
            part = paragraph[start:start + max_chars].strip()
            if len(part) >= 80:
                yield part

def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def normalized_fingerprint(text):
    return re.sub(r"\W+", "", text.casefold(), flags=re.UNICODE)

def shingle_signature(text, n=5):
    words = re.findall(r"\w+", text.casefold())
    if len(words) < n:
        return set()
    shingles = {" ".join(words[i:i+n]) for i in range(len(words)-n+1)}
    return {int(hashlib.blake2b(s.encode(), digest_size=8).hexdigest(), 16) % 100003
            for s in shingles}

In [ ]:
# 7. Candidate scan, per-source metrics, exact dedup, conservative near-dup signals
# Source-level dedup keeps the first source (sorted key) as canonical.
source_texts = {}
audit_rows = []
first_exact = {}
all_segments = {}
for key, info in sorted(records.items()):
    path = Path(info["path"])
    parts = []
    for doc in read_documents(path):
        parts.extend(segments(clean_text(doc)))
    unique = []
    within_dups = 0
    cross_dups = 0
    for seg in parts:
        h = digest(seg)
        if h in first_exact:
            if first_exact[h] == key:
                within_dups += 1
            else:
                cross_dups += 1
            continue
        first_exact[h] = key
        unique.append(seg)
    source_texts[key] = unique
    all_segments[key] = len(parts)
    # Token count is for candidate inventory, NOT authorization.
    count = sum(len(tok.encode(s, add_special_tokens=False)) for s in unique)
    audit_rows.append({
        "source": key, "lang": info["lang"], "domain": info["domain"],
        "segments_raw": len(parts), "segments_unique": len(unique),
        "duplicate_within": within_dups, "duplicate_cross_source": cross_dups,
        "characters_unique": sum(map(len, unique)),
        "tokens_proxy": count, "eligible": key in approved,
        "quality_flag": "few_segments" if len(unique) < 3 else "",
        "path": info["path"]
    })
df = pd.DataFrame(audit_rows)
df.to_csv(REPORT / "source_inventory_v6.csv", index=False)
print(df.groupby(["lang", "domain"], as_index=False).agg(
    sources=("source","count"), tokens_proxy=("tokens_proxy","sum"),
    eligible_sources=("eligible","sum")).to_string(index=False))
print("Potential:", int(df.tokens_proxy.sum()), "Approved:", int(df.loc[df.eligible, "tokens_proxy"].sum()))

In [ ]:
# 8. Near-duplicate source candidates (MinHash-style sample signature; review manually)
# Diagnostic only: never auto-remove source pairs based on this coarse sample.
def source_signature(segs, max_chars=20000):
    text = "\n".join(segs)[:max_chars]
    return shingle_signature(text)
signatures = {k: source_signature(v) for k,v in source_texts.items()}
pairs = []
keys = sorted(signatures)
for i, a in enumerate(keys):
    for b in keys[i+1:]:
        sa, sb = signatures[a], signatures[b]
        if not sa or not sb:
            continue
        jaccard = len(sa & sb) / len(sa | sb)
        if jaccard >= 0.50:
            pairs.append({"source_a": a, "source_b": b, "sample_jaccard": round(jaccard, 4)})
pd.DataFrame(pairs, columns=["source_a","source_b","sample_jaccard"]).to_csv(
    REPORT / "near_duplicate_candidates_v6.csv", index=False)
print("Potential near-duplicate source pairs:", len(pairs))

## C. Benchmark overlap audit

Optional: put held-out evaluation prompts in `corpus_v6/imports_benchmark/benchmark.jsonl` with `{"text":"..."}` per line. This audit checks exact normalized prompt presence in candidate training text; it **does not prove absence of semantic or paraphrase leakage**. If no benchmark is provided, readiness remains blocked.

In [ ]:
# 9. Optional benchmark exact overlap
benchmark_file = V6 / "imports_benchmark/benchmark.jsonl"
benchmark_items = []
if benchmark_file.exists():
    for item in read_documents(benchmark_file):
        if item.strip():
            benchmark_items.append(item.strip())
overlaps = []
if benchmark_items:
    approved_text = normalized_fingerprint("\n".join(
        s for k in sorted(approved) for s in source_texts[k]))
    for i, item in enumerate(benchmark_items):
        normalized = normalized_fingerprint(item)
        if normalized and normalized in approved_text:
            overlaps.append({"benchmark_index": i, "text_excerpt": item[:140]})
pd.DataFrame(overlaps, columns=["benchmark_index","text_excerpt"]).to_csv(
    REPORT / "benchmark_overlap_v6.csv", index=False)
print("Benchmark prompts:", len(benchmark_items), "| Exact normalized overlaps:", len(overlaps))

## D. Source-level splitting and rights-gated export

Every approved source belongs to exactly one split. All unapproved sources remain **inventory-only**. For small collections, per-language/domain cells may not be represented in every split; the manifest flags this. No synthetic or external material is automatically relabeled as authentic dialogue.

In [ ]:
# 10. Deterministic source-level splits, grouped by language/domain
SEED = 20261018
rng = random.Random(SEED)
split_map = {}
grouped = defaultdict(list)
for key in approved:
    grouped[(records[key]["lang"], records[key]["domain"])].append(key)
for cell, group in sorted(grouped.items()):
    group = sorted(group)
    rng.shuffle(group)
    n = len(group)
    if n < 3:
        # Conservative: cannot produce independent train/val/test in this cell.
        for key in group:
            split_map[key] = "train"
        continue
    n_test = max(1, round(n * 0.10))
    n_val = max(1, round(n * 0.10))
    for i, key in enumerate(group):
        split_map[key] = "test" if i < n_test else ("validation" if i < n_test+n_val else "train")
print("Split counts:", dict(Counter(split_map.values())))
print("Cells with <3 independent approved sources:",
      [f"{l}/{d}: {len(g)}" for (l,d),g in grouped.items() if len(g) < 3])

In [ ]:
# 11. Export only approved source segments, with provenance
import gzip
export_counts = {}
for split in ("train", "validation", "test"):
    target = EXPORT / f"{split}.jsonl.gz"
    count = 0
    with gzip.open(target, "wt", encoding="utf-8") as handle:
        for key in sorted(split_map):
            if split_map[key] != split:
                continue
            for idx, segment in enumerate(source_texts[key]):
                row = {"text": segment, "source": key, "lang": records[key]["lang"],
                       "domain": records[key]["domain"], "segment_id": idx,
                       "sha256": digest(segment)}
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
                count += 1
    export_counts[split] = count
print("Approved-only exports:", export_counts)

## E. Final tokenizer — optional experiment

Only run after **enough diverse sources are approved**. This experiment trains BPE tokenizers with 32k/40k/50k vocabulary targets from approved **training** data, using a bounded sample. Compare fertility (tokens per 1,000 characters) by language, coverage, and special-token handling before selecting a final tokenizer. Do not treat the proxy counts above as certified.

In [ ]:
# 12. Optional tokenizer comparison (OFF by default)
RUN_TOKENIZER_EXPERIMENT = False
if RUN_TOKENIZER_EXPERIMENT:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
    training_texts = [s for k in sorted(split_map) if split_map[k] == "train"
                      for s in source_texts[k]]
    if len(training_texts) < 1000 or len({records[k]["lang"] for k in split_map
                                         if split_map[k] == "train"}) < 3:
        raise RuntimeError("Insufficient approved, multilingual training data.")
    # Deterministic capped sampling prevents a single long source dominating the experiment.
    sample_rng = random.Random(SEED)
    sample_rng.shuffle(training_texts)
    training_texts = training_texts[:200000]
    results = []
    for vocab in (32000, 40000, 50000):
        t = Tokenizer(models.BPE(unk_token="[UNK]"))
        t.normalizer = normalizers.NFC()
        t.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        t.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(vocab_size=vocab, min_frequency=2,
                                      special_tokens=["[PAD]","[UNK]","[BOS]","[EOS]"])
        t.train_from_iterator(training_texts, trainer=trainer)
        t.save(str(REPORT / f"maia_bpe_candidate_{vocab}.json"))
        for lang in ("pt","en","es"):
            sample = [s for k in sorted(split_map) if split_map[k] == "train"
                      and records[k]["lang"] == lang for s in source_texts[k][:100]]
            chars = sum(len(s) for s in sample)
            ntokens = sum(len(t.encode(s).ids) for s in sample)
            results.append({"vocab_target": vocab, "vocab_actual": t.get_vocab_size(),
                            "lang": lang, "tokens_per_1000_chars":
                            round(1000 * ntokens / max(chars, 1), 2)})
    pd.DataFrame(results).to_csv(REPORT / "tokenizer_comparison_v6.csv", index=False)
    print(pd.DataFrame(results).to_string(index=False))
else:
    print("Tokenizer experiment skipped. Set RUN_TOKENIZER_EXPERIMENT=True after review.")

In [ ]:
# 13. Reproducible manifest and explicit readiness gates
TARGET_LANGUAGE = {"pt": .40, "en": .35, "es": .25}
TARGET_DOMAIN = {"language": .30, "dialogue": .20, "logic": .25,
                 "educational": .15, "creative": .10}
eligible_df = df[df.eligible]
eligible_tokens = int(eligible_df.tokens_proxy.sum())
potential_tokens = int(df.tokens_proxy.sum())
approved_cells = {(r.lang, r.domain) for r in eligible_df.itertuples()}
missing_cells = [f"{lang}/{domain}" for lang in TARGET_LANGUAGE
                 for domain in TARGET_DOMAIN if (lang,domain) not in approved_cells]
auth_dialogue = int(eligible_df[eligible_df.domain == "dialogue"].tokens_proxy.sum())
weak_cells = [f"{l}/{d}" for (l,d),g in grouped.items() if len(g) < 3]
blockers = []
if len(approved) < len(records):
    blockers.append("External rights review incomplete")
if missing_cells:
    blockers.append("Approved language/domain cells missing")
if auth_dialogue < 100000:
    blockers.append("Fewer than 100k reviewed authentic dialogue proxy tokens; verify provenance")
if len(approved) < 3 or weak_cells:
    blockers.append("Independent source-level train/validation/test unavailable for some cells")
if not RUN_TOKENIZER_EXPERIMENT:
    blockers.append("Final Maia tokenizer not trained; token balance not certified")
if not benchmark_items:
    blockers.append("Benchmark leakage audit missing: no benchmark supplied")
if overlaps:
    blockers.append("Benchmark exact overlaps found")
blockers.append("Near-duplicate and quality diagnostics require human review")
report = {
    "version": "v6", "generated_utc": datetime.now(timezone.utc).isoformat(),
    "source_manifest_v5": str(V5_MANIFEST),
    "tokenizer_for_audit": "gpt2 (proxy)",
    "potential_external_tokens_proxy": potential_tokens,
    "approved_external_tokens_proxy": eligible_tokens,
    "source_count": len(records), "approved_source_count": len(approved),
    "source_level_splits": split_map,
    "split_segments": export_counts,
    "missing_approved_cells": missing_cells,
    "weak_independent_source_cells": weak_cells,
    "benchmark_items": len(benchmark_items), "benchmark_exact_overlaps": len(overlaps),
    "near_duplicate_candidate_pairs": len(pairs),
    "target_language": TARGET_LANGUAGE, "target_domain": TARGET_DOMAIN,
    "acquisition_errors_v5": len(manifest5.get("acquisition_errors", [])),
    "readiness_blockers": blockers, "production_ready": False,
    "files": {}
}
for p in sorted(REPORT.glob("*")) + sorted(EXPORT.glob("*")):
    if p.is_file():
        report["files"][p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
MANIFEST = REPORT / "maia_corpus_v030_manifest_v6.json"
MANIFEST.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Manifest:", MANIFEST)
print("PRODUCTION READY:", report["production_ready"])
print("Blockers:")
for item in blockers:
    print(" -", item)

## F. Optional polite MediaWiki downloader (not run by default)

Use only for **small, explicitly selected page batches**. A 429 means slow down or stop. Honor `Retry-After`, identify the application and provide a contact URL/email in the User-Agent. Do **not** evade rate limits. For bulk acquisition prefer Wikimedia dumps or official datasets. Downloaded content remains `pending_review`.

In [ ]:
# 14. Optional single-request helper (not invoked automatically)
import email.utils
from datetime import datetime, timezone
import requests
def polite_mediawiki_get(api_url, params, user_agent, max_attempts=4):
    if "contact" in user_agent.lower() or "example" in user_agent.lower():
        raise ValueError("Set a real descriptive User-Agent with contact information.")
    session = requests.Session()
    session.headers.update({"User-Agent": user_agent, "Accept": "application/json"})
    for attempt in range(max_attempts):
        response = session.get(api_url, params=params, timeout=35)
        if response.status_code == 429:
            if attempt == max_attempts - 1:
                response.raise_for_status()
            raw = response.headers.get("Retry-After", "")
            try:
                wait = max(0, min(300, int(raw)))
            except ValueError:
                try:
                    parsed = email.utils.parsedate_to_datetime(raw)
                    wait = max(0, min(300, (parsed - datetime.now(timezone.utc)).total_seconds()))
                except (TypeError, ValueError, OverflowError):
                    wait = min(120, 5 * (2 ** attempt))
            print(f"429: waiting {wait:.1f}s before retry")
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    raise RuntimeError("No successful response")
# Example (edit UA and page title before using):
# polite_mediawiki_get("https://pt.wikibooks.org/w/api.php",
#     {"action":"query","prop":"extracts","explaintext":1,"titles":"...", "format":"json"},
#     "MaiaCorpusResearch/0.3 (researcher@your-real-domain.org)")

## Interpretation

**Do not start production pretraining from this notebook alone.** Review rights, source independence, quality, benchmark leakage and domain coverage. When the review JSON is edited, rerun from Cell 5 onward. When new source files are imported, rerun from Cell 3 onward.

The readiness flag is deliberately conservative: it remains false until the audits and governance requirements are independently certified. The optional tokenizer experiment is diagnostic, not a finalized model tokenizer.